# Saber Dynamic n/mu/floor Sweep — GSM8K (LIMIT=150)

动态调整 n / mu / remask_floor，实现"先乱写、后精修"：
- n_t: 早期大（敢写），后期小（谨慎）
- mu_t: 早期小（敢退），后期大（保守）
- floor_t: 早期高（退回下界高），后期低（趋近0）

进度指标：m = window 内 mask 比例（1.0=全 mask，0.0=全解完）

30 个任务：1 anchor + 14 保守版 + 15 激进版

## 1. 环境设置

In [ ]:
import os, gc, re, json, datetime, threading, queue, subprocess
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '4,5')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)
os.makedirs('evals_results/saber', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

## 2. 任务配置

In [ ]:
task = 'gsm8k'
fewshot = 5
seed = 42
gen_length = 256
steps = 256
limit_samples = 150
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')


def _dyn(name, n_hi, n_lo, mu_lo, mu_hi, gamma_n, gamma_mu, gamma_floor,
         remask_ratio_hi, floor_lo, berm_scope='trail_block'):
    return {
        'name': name,
        'extra_args': [
            'saber_dynamic=True',
            'berm_mode=cross_step',
            'saber_global_aadu=True',
            'saber_mtr=0.8',
            'block_length=32',
            f'saber_n_hi={n_hi}',
            f'saber_n_lo={n_lo}',
            f'saber_mu_lo={mu_lo}',
            f'saber_mu_hi={mu_hi}',
            f'saber_gamma_n={gamma_n}',
            f'saber_gamma_mu={gamma_mu}',
            f'saber_gamma_floor={gamma_floor}',
            f'saber_remask_ratio_hi={remask_ratio_hi}',
            f'saber_floor_lo={floor_lo}',
            f'saber_berm_scope={berm_scope}',
        ],
    }


def _anchor(name, n, mu):
    return {
        'name': name,
        'extra_args': [
            'saber_expand=True',
            'berm_mode=cross_step',
            'saber_global_aadu=True',
            'saber_mtr=0.8',
            f'saber_n={n}',
            f'saber_mu={mu}',
            'block_length=32',
        ],
    }


# C = conservative, A = aggressive
TASK_CONFIGS = [
    # anchor (static baseline)
    _anchor('dyn_anchor_n4_mu8', 4, 8),

    # ── Conservative (C): n_hi=6, n_lo=3, mu_lo=6, mu_hi=16, rr=0.25, fl=1 ──
    _dyn('dyn_C_g10', 6, 3, 6, 16, 1.0, 1.0, 1.5, 0.25, 1),
    _dyn('dyn_C_g15', 6, 3, 6, 16, 1.5, 1.5, 1.5, 0.25, 1),
    _dyn('dyn_C_g08', 6, 3, 6, 16, 0.8, 0.8, 1.5, 0.25, 1),
    _dyn('dyn_C_g05', 6, 3, 6, 16, 0.5, 0.5, 1.5, 0.25, 1),
    _dyn('dyn_C_nhi8', 8, 3, 6, 16, 1.0, 1.0, 1.5, 0.25, 1),
    _dyn('dyn_C_nhi7', 7, 3, 6, 16, 1.0, 1.0, 1.5, 0.25, 1),
    _dyn('dyn_C_nlo2', 6, 2, 6, 16, 1.0, 1.0, 1.5, 0.25, 1),
    _dyn('dyn_C_nlo4', 6, 4, 6, 16, 1.0, 1.0, 1.5, 0.25, 1),
    _dyn('dyn_C_muhi24', 6, 3, 6, 24, 1.0, 1.0, 1.5, 0.25, 1),
    _dyn('dyn_C_mulo4', 6, 3, 4, 16, 1.0, 1.0, 1.5, 0.25, 1),
    _dyn('dyn_C_rr40', 6, 3, 6, 16, 1.0, 1.0, 1.5, 0.40, 1),
    _dyn('dyn_C_rr15', 6, 3, 6, 16, 1.0, 1.0, 1.5, 0.15, 1),
    _dyn('dyn_C_gf10', 6, 3, 6, 16, 1.0, 1.0, 1.0, 0.25, 1),
    _dyn('dyn_C_window', 6, 3, 6, 16, 1.0, 1.0, 1.5, 0.25, 1, 'window'),

    # ── Aggressive (A): n_hi=8, n_lo=3, mu_lo=4, mu_hi=16, rr=0.35, fl=0 ──
    _dyn('dyn_A_g10', 8, 3, 4, 16, 1.0, 1.0, 1.5, 0.35, 0),
    _dyn('dyn_A_g12', 8, 3, 4, 16, 1.2, 1.2, 1.5, 0.35, 0),
    _dyn('dyn_A_g20', 8, 3, 4, 16, 2.0, 2.0, 1.5, 0.35, 0),
    _dyn('dyn_A_g05', 8, 3, 4, 16, 0.5, 0.5, 1.5, 0.35, 0),
    _dyn('dyn_A_nhi10', 10, 3, 4, 16, 1.2, 1.2, 1.5, 0.35, 0),
    _dyn('dyn_A_nhi12', 12, 3, 4, 16, 1.2, 1.2, 1.5, 0.35, 0),
    _dyn('dyn_A_nlo4', 8, 4, 4, 16, 1.2, 1.2, 1.5, 0.35, 0),
    _dyn('dyn_A_nlo2', 8, 2, 4, 16, 1.2, 1.2, 1.5, 0.35, 0),
    _dyn('dyn_A_muhi24', 8, 3, 4, 24, 1.2, 1.2, 1.5, 0.35, 0),
    _dyn('dyn_A_mulo2', 8, 3, 2, 16, 1.2, 1.2, 1.5, 0.35, 0),
    _dyn('dyn_A_rr50', 8, 3, 4, 16, 1.2, 1.2, 1.5, 0.50, 0),
    _dyn('dyn_A_rr20', 8, 3, 4, 16, 1.2, 1.2, 1.5, 0.20, 0),
    _dyn('dyn_A_gf10', 8, 3, 4, 16, 1.2, 1.2, 1.0, 0.35, 0),
    _dyn('dyn_A_window', 8, 3, 4, 16, 1.2, 1.2, 1.5, 0.35, 0, 'window'),
    _dyn('dyn_A_curblk', 8, 3, 4, 16, 1.2, 1.2, 1.5, 0.35, 0, 'current_block'),
]

GPU_POOL = [int(x) for x in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]
print(f'Total tasks: {len(TASK_CONFIGS)} | GPUs: {GPU_POOL} | limit: {limit_samples} | timestamp: {timestamp}')

## 3. 并行启动任务

In [ ]:
task_queue = queue.Queue()
for cfg in TASK_CONFIGS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_results = []


def gpu_worker(gpu_id):
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = cfg['name']
        log_file = f'nlogs/sweep_dynamic_{task}_{name}_{timestamp}.log'
        output_dir = f'evals_results/saber/{task}-{name}-{timestamp}'

        common_args = [
            "model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f'gen_length={gen_length}',
            f'steps={steps}',
            'show_speed=True',
            f'seed={seed}',
        ]
        model_args = ','.join(common_args + cfg['extra_args'])
        cmd = (
            f'CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py '
            f'--tasks {task} --num_fewshot {fewshot} --confirm_run_unsafe_code '
            f'--model llada_dist --model_args {model_args} --output_path {output_dir} '
            f'--log_samples --limit {limit_samples}'
        )

        print(f'[GPU {gpu_id}] START {name}')
        p = subprocess.Popen(
            cmd,
            shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()
        print(f'[GPU {gpu_id}] DONE  {name} rc={rc}')
        with results_lock:
            all_results.append((cfg, name, log_file, output_dir, rc))
        task_queue.task_done()


threads = []
for gid in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gid,), daemon=True)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print('All finished:', len(all_results), '/', len(TASK_CONFIGS))

## 4. 解析评测结果

In [ ]:
def parse_result(cfg, output_dir, log_file):
    name = cfg['name']

    result_json = Path(output_dir) / 'results.json'
    flex_acc = None
    strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')

    log_content = Path(log_file).read_text(encoding='utf-8', errors='ignore') if Path(log_file).exists() else ''

    if flex_acc is None:
        m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', log_content)
        if m: flex_acc = float(m.group(1))
    if strict_acc is None:
        m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', log_content)
        if m: strict_acc = float(m.group(1))

    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', log_content)
    if speed_m is None:
        speed_m = re.search(r'Average generation speed:\s*([\d.]+)', log_content)
    nfe_m = re.search(r'Total NFE is (\d+)', log_content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', log_content)
    if tok_m is None:
        tok_m = re.search(r'Total tokens generated:\s*(\d+)', log_content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', log_content)
    if time_m is None:
        time_m = re.search(r'Total generation time:\s*([\d.]+)', log_content)

    variant = 'anchor' if 'anchor' in name else ('C' if '_C_' in name else 'A')

    return {
        'name': name,
        'variant': variant,
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    }


rows = []
for cfg, name, log_file, output_dir, rc in all_results:
    rows.append(parse_result(cfg, output_dir, log_file))

df = pd.DataFrame(rows).sort_values(['variant', 'name'])
pd.set_option('display.max_rows', 50)

print(f'\n{"Name":<26} {"Type":<7} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10} {"Time(s)":<8}')
print('-' * 82)
for _, r in df.iterrows():
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(int(r['total_nfe'])) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<26} {r['variant']:<7} {fa:<10} {sa:<11} {sp:<10} {nf:<10} {tm:<8}")

## 5. 对比图

In [ ]:
plot_df = df[df['flex_acc'].notna()].copy()
plot_df['label'] = plot_df['name']

fig, axes = plt.subplots(3, 1, figsize=(18, 13), sharex=True)
metrics = [('flex_acc', 'FlexAcc'), ('total_nfe', 'Total NFE'), ('tok_per_sec', 'Tokens/sec')]

var_colors = {'anchor': '#E53935', 'C': '#4E79A7', 'A': '#59A14F'}
colors = [var_colors.get(r['variant'], '#999') for _, r in plot_df.iterrows()]

for ax, (col, title) in zip(axes, metrics):
    vals = plot_df[col].fillna(0)
    ax.bar(plot_df['label'], vals, color=colors)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

axes[-1].tick_params(axis='x', rotation=65, labelsize=7)
plt.tight_layout()
plt.show()

## 6. 从已有结果重新加载（可选）

内核重启后运行此 cell + Section 5。

In [ ]:
import glob, re, json, os
from pathlib import Path
import pandas as pd

task = 'gsm8k'

NAMES = [
    'dyn_anchor_n4_mu8',
    'dyn_C_g10',
    'dyn_C_g15',
    'dyn_C_g08',
    'dyn_C_g05',
    'dyn_C_nhi8',
    'dyn_C_nhi7',
    'dyn_C_nlo2',
    'dyn_C_nlo4',
    'dyn_C_muhi24',
    'dyn_C_mulo4',
    'dyn_C_rr40',
    'dyn_C_rr15',
    'dyn_C_gf10',
    'dyn_C_window',
    'dyn_A_g10',
    'dyn_A_g12',
    'dyn_A_g20',
    'dyn_A_g05',
    'dyn_A_nhi10',
    'dyn_A_nhi12',
    'dyn_A_nlo4',
    'dyn_A_nlo2',
    'dyn_A_muhi24',
    'dyn_A_mulo2',
    'dyn_A_rr50',
    'dyn_A_rr20',
    'dyn_A_gf10',
    'dyn_A_window',
    'dyn_A_curblk',
]

latest = sorted(
    glob.glob(f'nlogs/sweep_dynamic_{task}_dyn_anchor_n4_mu8_*.log'),
    key=os.path.getmtime, reverse=True,
)
if latest:
    fname = os.path.basename(latest[0])
    timestamp = fname.replace(f'sweep_dynamic_{task}_dyn_anchor_n4_mu8_', '').replace('.log', '')
    print(f'Auto-detected timestamp: {timestamp}')
else:
    timestamp = 'NOTFOUND'
    print('WARNING: No log found!')

parsed_results = []
for name in NAMES:
    lf = f'nlogs/sweep_dynamic_{task}_{name}_{timestamp}.log'
    od = f'evals_results/saber/{task}-{name}-{timestamp}'
    if not os.path.exists(lf):
        print(f'  [MISS] {name}')
        continue
    log_content = Path(lf).read_text(encoding='utf-8', errors='ignore')
    result_json = Path(od) / 'results.json'
    flex_acc = strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')
    if flex_acc is None:
        m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', log_content)
        if m: flex_acc = float(m.group(1))
    if strict_acc is None:
        m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', log_content)
        if m: strict_acc = float(m.group(1))
    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', log_content)
    if speed_m is None: speed_m = re.search(r'Average generation speed:\s*([\d.]+)', log_content)
    nfe_m = re.search(r'Total NFE is (\d+)', log_content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', log_content)
    if tok_m is None: tok_m = re.search(r'Total tokens generated:\s*(\d+)', log_content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', log_content)
    if time_m is None: time_m = re.search(r'Total generation time:\s*([\d.]+)', log_content)
    variant = 'anchor' if 'anchor' in name else ('C' if '_C_' in name else 'A')
    parsed_results.append({
        'name': name,
        'variant': variant,
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    })
    print(f'  [OK]   {name}')

df = pd.DataFrame(parsed_results).sort_values(['variant', 'name'])
print(f'\nLoaded {len(parsed_results)} results (timestamp={timestamp})')
print(f'\n{"Name":<26} {"Type":<7} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10} {"Time(s)":<8}')
print('-' * 82)
for _, r in df.iterrows():
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(int(r['total_nfe'])) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<26} {r['variant']:<7} {fa:<10} {sa:<11} {sp:<10} {nf:<10} {tm:<8}")

print(f'\nRe-run Section 5 to regenerate plots.')